
This Colab notebook provides an example of fine-tuning ConfliBERT for binary classification. It will run using free Colab resources. For more information on ConfliBERT please visit https://eventdata.utdallas.edu/. See our other Colab for more concepts and general applications: ConfliBERT Concepts and Applications.

This material is based on work supported by the National Science Foundation under Grant No. OAC-2311142. Any opinions, findings, and conclusions or recommendations expressed in this material are those of the author(s) and do not necessarily reflect the views of the National Science Foundation.

# Peace Science Participants:

1.   Do: File -> Save a copy in Drive
2.   Switch tabs to the copied file
3.   Check: File -> Locate in Drive
4.   (optional) Rename the file

# Getting Set Up

Connect a runtime with a GPU for optimal performance:

1.   Click on Runtime -> Change runtime type
2.   Make sure you're running Python 3
3.   Under Hardware accelerator, select T4 GPU. This will work for several runs using free Colab resources.
4.   Click on "Connect" in the top right.

In [16]:
# Let's check out what we're using

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Mar 15 03:05:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             31W /   70W |    2945MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Install packages and clone github source code.

**Note:** when executing Code in Colab, you run shell commands using the !. Except when changing directories, for that you use %, which is a special command for Colab. Otherwise, default execution assumes Python 3.

In [1]:
# Clone the source code
# Can go to our ConfliBERT github
!git clone https://github.com/eventdata/ConfliBERT.git

Cloning into 'ConfliBERT'...
remote: Enumerating objects: 709, done.
remote: Counting objects: 100% (284/284), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 709 (delta 250), reused 201 (delta 187), pack-reused 425 (from 2)
Receiving objects: 100% (709/709), 28.60 MiB | 15.49 MiB/s, done.
Resolving deltas: 100% (348/348), done.


In [17]:
# changes directory to ConfliBERT, which was just copied from github
# click the Files folder on the left to see and compare to https://github.com/eventdata/ConfliBERT
%cd /content/ConfliBERT

/content


In [18]:
# verify the working directory is /content/ConfliBERT
!pwd

/content/ConfliBERT


In [2]:
import os
os.chdir('ConfliBERT')
!git checkout dev

Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'


In [3]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.8 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=087f4ba8917be0fa79ae870fb9c17d97a08cfe14c6e3385ec39c93715cf7e4f6
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Success

In [4]:
import argparse
import pandas as pd
import os
import json
import torch
import torch.nn as nn
import numpy as np
from finetune_data import train_multi_seed,loadData
from ipywidgets import widgets
from IPython.display import display
from finetune_data import train_multi_seed,loadData
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [5]:
!pip freeze >> packages.txt

# Define datasets and training parameters.

You can select the available processed dataset in the repo and its corresponding task:

| Dataset | Task |
| :-------- | :-------- |
|20news| binary |
| BBC_News | binary |
| IndiaPoliceEvents_doc | multilabel |
| IndiaPoliceEvents_sents | multilabel|
| cameo_class | multiclass |
| cameo_ner | ner |
| insightCrime | multilabel |
| re3d | ner |
| satp_relevant | multilabel|



See more details at https://github.com/eventdata/ConfliBERT/tree/main/configs

In [6]:
# create and set up the 'args' object
# this holds the values we're passing off to simpletransformers
# the call to simpletransformers happens in finetune_data.py
# https://simpletransformers.ai/docs/usage/

args = argparse.Namespace()

# Set dataset
args.dataset = "BBC_News"
args.report_per_epoch= True

# Set training configurations.
training_configs = \
{
    "task": "binary",   # task should match your choosen dataset
    "num_of_seeds": 1, # would use multiple seeds with more resources
    "initial_seed": 123,
    "epochs_per_seed": 2,   # we only train 2 epochs in this demo to reduce training time. that means the model sees the data twice, updating as it goes
    "train_batch_size": 16, # can try 8 or 32, it controls the amount of data used to update the model per update
    "max_seq_length": 128, # don't change this one, these models use smaller sequences, some new models can use larger sequences. the sequence is where the transformer operates
    "models": [             # We include one model in this example. You can include as many as you like.
        {
            "model_name": "ConfliBERT-scr-uncased",
            "model_path": "snowood1/ConfliBERT-scr-uncased", # simpletransformers gets this model from huggingface
            "architecture": "bert",
            "do_lower_case": True # should be true if using an uncased model
        } # , this part is commented out so it won't execute, but this is how you'd add additional models
        #{
        #    "model_name": "bert-base-uncased",
        #    "model_path": "bert-base-uncased",
        #    "architecture": "bert",
        #    "do_lower_case": True
        #}
    ]
}

## Saving the custom configuration.
for k,v in training_configs.items():
    setattr(args, k, v)

args.data_dir = os.path.join("./data/", args.dataset, "")

## Loading the datasets. loadData is a function in finetune_data.py
train_df, eval_df, test_df, args.num_labels = loadData(args)

if args.task == "ner":
    with open(os.path.join(args.data_dir, "labels.json")) as json_file:
        args.labels_list = json.load(json_file)

# View of training data

In [19]:
train_df.head()

,text,labels
0,tv future in the hands of viewers with home th...,0
1,worldcom boss left books alone former worldc...,0
2,tigers wary of farrell gamble leicester say ...,0
3,yeading face newcastle in fa cup premiership s...,0
4,howard hits back at mongrel jibe michael howar...,1


# Training

In [7]:
## Running experiments for all the models in configs:
for model_configs in args.models:

    # args.output_dir = os.path.join("./outputs/", args.dataset + "_" + model_configs["model_name"], "")
    args.output_dir = os.path.join("./outputs/", args.dataset, "")

    train_multi_seed(args, train_df, eval_df, test_df, model_configs)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# View evaluation logs

In [8]:
df = pd.read_csv(f"./logs/{args.dataset}_full_report.csv")
df

,tp,tn,fp,fn,acc,prec,rec,f1,data_name,model_name,seed,train_batch_size,epoch
0,53,264,5,0,0.984472,0.913793,1.000000,0.954955,BBC_News,ConfliBERT-scr-uncased,123,16,1
1,52,267,2,1,0.990683,0.962963,0.981132,0.971963,BBC_News,ConfliBERT-scr-uncased,123,16,2


In [11]:
# during training we periodically check performance, this is selecting the best of those
model_path = "/content/ConfliBERT/outputs/BBC_News/best_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [20]:
# have a look at the test data
test_df.head()

,text,labels,predicted,probability0,probability1
0,ocean s twelve raids box office ocean s twelve...,0,Not Political,0.998882,0.001118
1,wilkinson fit to face edinburgh england captai...,0,Not Political,0.998882,0.001118
2,security warning over fbi virus the us feder...,0,Not Political,0.998882,0.001118
3,moya fights back for indian title carlos moya ...,0,Not Political,0.998882,0.001118
4,disappointed scott in solid start allan scott ...,0,Not Political,0.998882,0.001118


In [13]:
# Makes predictions on every row of a column 'text'
# Adds columns with the binary prediction (political news or not) and probabilities corresponding to each number

i = 0

class_map = {0: "Not Political", 1: "Political"}
softmax = nn.Softmax(dim=1)

for index, row in test_df.iterrows():
    text = row['text']

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = softmax(logits).cpu()
    prediction = logits.argmax(-1).item()

    test_df.at[index, 'predicted'] = class_map.get(prediction, 'Unknown')
    test_df.at[index, 'probability0'] = probs[0, 0].item()
    test_df.at[index, 'probability1'] = probs[0, 1].item()
    i += 1
    print(i)

test_df.to_csv('/content/ConfliBERT/outputs/BBC_News/results.tsv', sep='\t')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [15]:
test_df.head()

,text,labels,predicted,probability0,probability1
0,ocean s twelve raids box office ocean s twelve...,0,Not Political,0.998882,0.001118
1,wilkinson fit to face edinburgh england captai...,0,Not Political,0.998882,0.001118
2,security warning over fbi virus the us feder...,0,Not Political,0.998882,0.001118
3,moya fights back for indian title carlos moya ...,0,Not Political,0.998882,0.001118
4,disappointed scott in solid start allan scott ...,0,Not Political,0.998882,0.001118
